In [2]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("SupplyChain-EDA") \
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

df = spark.read.format("delta").load("../lakehouse/bronze/orders_raw")
print(f"Rows: {df.count():,}")
print(f"Columns: {len(df.columns)}")

your 131072x1 screen size is bogus. expect trouble
26/04/08 18:32:02 WARN Utils: Your hostname, MarioLegion resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/04/08 18:32:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Ivy Default Cache set to: /home/huevacles/.ivy2/cache
The jars for the packages stored in: /home/huevacles/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ceae8d30-13a4-47a5-84a1-f0dccbd6f33f;1.0
	confs: [default]


:: loading settings :: url = jar:file:/home/huevacles/dev/supply-chain-analytics-databricks/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 116ms :: artifacts dl 5ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-ceae8d30-13a4-47a5-84a1-f0dccbd6f33f
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/2ms)
26/04/08 18:32:02

Rows: 180,519
Columns: 53


In [3]:
df.printSchema()


root
 |-- type: string (nullable = true)
 |-- days_for_shipping_real: integer (nullable = true)
 |-- days_for_shipment_scheduled: integer (nullable = true)
 |-- benefit_per_order: double (nullable = true)
 |-- sales_per_customer: double (nullable = true)
 |-- delivery_status: string (nullable = true)
 |-- late_delivery_risk: integer (nullable = true)
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- customer_city: string (nullable = true)
 |-- customer_country: string (nullable = true)
 |-- customer_email: string (nullable = true)
 |-- customer_fname: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- customer_lname: string (nullable = true)
 |-- customer_password: string (nullable = true)
 |-- customer_segment: string (nullable = true)
 |-- customer_state: string (nullable = true)
 |-- customer_street: string (nullable = true)
 |-- customer_zipcode: integer (nullable = true)
 |-- department_id: integer (nullable = true

In [4]:
null_counts = df.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in df.columns
]).toPandas().T

null_counts.columns = ["null_count"]
null_counts["pct"] = (null_counts["null_count"] / df.count() * 100).round(2)
null_counts[null_counts["null_count"] > 0]


,null_count,pct
customer_lname,8,0.00
customer_zipcode,3,0.00
order_zipcode,155679,86.24
product_description,180519,100.00


In [5]:
total = df.count()
distinct = df.dropDuplicates(["order_item_id"]).count()
print(f"Total rows               : {total:,}")
print(f"Distinct by order_item_id: {distinct:,}")
print(f"Duplicates               : {total - distinct:,}")

Total rows               : 180,519
Distinct by order_item_id: 180,519
Duplicates               : 0


In [6]:
df.select(
    "days_for_shipping_real",
    "days_for_shipment_scheduled",
    "benefit_per_order",
    "sales",
    "order_item_profit_ratio"
).describe().show()

+-------+----------------------+---------------------------+------------------+------------------+-----------------------+
|summary|days_for_shipping_real|days_for_shipment_scheduled| benefit_per_order|             sales|order_item_profit_ratio|
+-------+----------------------+---------------------------+------------------+------------------+-----------------------+
|  count|                180519|                     180519|            180519|            180519|                 180519|
|   mean|    3.4976539865609713|          2.931846509231715|21.974988638594166| 203.7720960861701|    0.12064663549026358|
| stddev|    1.6237218283741637|         1.3744492800079782|104.43352574698666|132.27307749970336|     0.4667956046074935|
|    min|                     0|                          0|       -4274.97998|       9.989999771|                  -2.75|
|    max|                     6|                          4|       911.7999878|        1999.98999|                    0.5|
+-------+-------

In [7]:
for col in ["delivery_status", "shipping_mode", "market", "order_region", "customer_segment", "order_status"]:
    print(f"\n── {col} ──")
    df.groupBy(col).count().orderBy("count", ascending=False).show(truncate=False)


── delivery_status ──
+-----------------+-----+
|delivery_status  |count|
+-----------------+-----+
|Late delivery    |98977|
|Advance shipping |41592|
|Shipping on time |32196|
|Shipping canceled|7754 |
+-----------------+-----+


── shipping_mode ──
+--------------+------+
|shipping_mode |count |
+--------------+------+
|Standard Class|107752|
|Second Class  |35216 |
|First Class   |27814 |
|Same Day      |9737  |
+--------------+------+


── market ──
+------------+-----+
|market      |count|
+------------+-----+
|LATAM       |51594|
|Europe      |50252|
|Pacific Asia|41260|
|USCA        |25799|
|Africa      |11614|
+------------+-----+


── order_region ──
+---------------+-----+
|order_region   |count|
+---------------+-----+
|Central America|28341|
|Western Europe |27109|
|South America  |14935|
|Oceania        |10148|
|Northern Europe|9792 |
|Southeast Asia |9539 |
|Southern Europe|9431 |
|Caribbean      |8318 |
|West of USA    |7993 |
|South Asia     |7731 |
|Eastern Asia   |7

In [8]:
df.select(
    F.min("order_date_dateorders").alias("earliest_order"),
    F.max("order_date_dateorders").alias("latest_order")
).show(truncate=False)

+--------------+-------------+
|earliest_order|latest_order |
+--------------+-------------+
|1/1/2015 0:00 |9/9/2017 9:50|
+--------------+-------------+

